# 🔍 LINE 偽冒下載網站偵測工具

模擬使用者在 Google 搜尋「LINE 下載」，掃描前幾頁結果，
**自動篩選出非官方且含簡體中文的偽冒網站**。

| 步驟 | 說明 |
|------|------|
| 1 | 用多組關鍵字模擬 Google 搜尋 LINE 下載 |
| 2 | 取出搜尋結果 URL，排除官方網域 |
| 3 | 實際訪問各網站，偵測頁面是否含簡體中文 |
| 4 | 輸出黑名單清單並下載 |

▶ **「執行階段」→「全部執行」**

In [ ]:
# Cell 1｜安裝 Chrome + 套件

# Step 1：安裝 Chrome（Colab 不內建，必須手動裝）
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q 2>/dev/null
!rm google-chrome-stable_current_amd64.deb

# Step 2：安裝 Python 套件
!pip install requests beautifulsoup4 lxml selenium webdriver-manager -q

# Step 3：確認版本
!google-chrome --version
import sys
print(f'Python {sys.version[:6]} ✅')
print('所有套件安裝完成 ✅')


In [ ]:
# Cell 2｜設定與工具函式
import re, csv, json, time, urllib.parse, warnings
from datetime import datetime
from pathlib import Path
from collections import defaultdict
from IPython.display import display, HTML
import requests
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('/content/line_phishing_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 搜尋關鍵字（模擬使用者 Google 搜尋習慣）
SEARCH_QUERIES = [
    'LINE 下載',
    'LINE 電腦版 下載',
    'LINE PC版 下載',
    'LINE官方下載',
    'line download',
    'LINE app 免費下載',
    'LINE桌面版下載',
]

RESULTS_PER_QUERY = 20

# ── LINE 官方合法網域白名單（這些直接跳過）
OFFICIAL_DOMAINS = {
    'line.me', 'linecorp.com', 'line-apps.com', 'line-scdn.net',
    'naver.com', 'lycorp.co.jp', 'line-obj.com', 'lin.ee',
    'line.biz', 'linebank.com.tw', 'linepay.me',
    # 合法下載平台
    'apple.com', 'apps.apple.com', 'google.com', 'play.google.com',
    'microsoft.com', 'aka.ms', 'softonic.com', 'softonic.com.br',
    'wikipedia.org', 'youtube.com', 'facebook.com', 'instagram.com',
    'twitter.com', 'x.com', 'techradar.com', 'pcmag.com',
    'cnet.com', 'zdnet.com', 'techcrunch.com', 'macrumors.com',
    '9to5mac.com', 'androidauthority.com', 'apkmirror.com',
    'apkpure.com', 'uptodown.com', 'filehippo.com',
}

# ── 偽冒 LINE 網域特徵（只要包含這些模式就列為可疑）
# 邏輯：網域含「line」但不在官方白名單 → 可疑偽冒站
LINE_SPOOF_KEYWORDS = [
    'line',   # 最基本：含 line 字眼
]

# ── 簡體中文特有字集（偵測用，非必要條件，只作為額外標記）
SIMPLIFIED_ONLY_CHARS = set(
    '这来时说们为国会对产业进线应该从发现问题员间关实结'
    '无马处长动话与头联专认证验码账户钱转换请输入您的'
    '账号密码登录点击这里立即下载安装获取免费领取扫描'
    '绑定解除限制异常检测升级版本官方客服联系我们软件'
)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'zh-TW,zh;q=0.9,en;q=0.8',
}

# ────────────────────────────────────────────────
# 工具函式
# ────────────────────────────────────────────────

def extract_domain(url):
    try:
        parsed = urllib.parse.urlparse(url)
        return parsed.hostname.lower().lstrip('www.') if parsed.hostname else ''
    except:
        return ''

def is_official(domain):
    """官方白名單：完全匹配或子網域匹配"""
    return any(domain == od or domain.endswith('.' + od) for od in OFFICIAL_DOMAINS)

def is_line_spoof(domain):
    """
    判斷是否為偽冒 LINE 的可疑網域。
    條件：網域含 'line' 字眼，且不在官方白名單內。
    範例命中：linebbn.com / line-download.net / getline.app
    """
    return any(kw in domain for kw in LINE_SPOOF_KEYWORDS)

def has_simplified_chinese(text):
    """偵測是否含簡中（額外標記用，非必要條件）"""
    if not text: return False
    return any(ch in SIMPLIFIED_ONLY_CHARS for ch in text)

def get_simplified_hits(text):
    return list(dict.fromkeys(ch for ch in text if ch in SIMPLIFIED_ONLY_CHARS))[:8]

def fetch_page(url, timeout=10):
    """抓取頁面，回傳 (title, body_text, final_url)"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout,
                            allow_redirects=True, verify=False)
        resp.encoding = resp.apparent_encoding or 'utf-8'
        html = resp.text
        final_url = resp.url

        title_m = re.search(r'<title[^>]*>(.*?)</title>', html, re.IGNORECASE | re.DOTALL)
        title = re.sub(r'<[^>]+>', '', title_m.group(1)).strip() if title_m else ''

        desc_m = re.search(r'<meta[^>]+name=["\']description["\'][^>]+content=["\']([^"\']*)',
                           html, re.IGNORECASE)
        desc = desc_m.group(1) if desc_m else ''

        body_m = re.search(r'<body[^>]*>(.*)', html[:8000], re.IGNORECASE | re.DOTALL)
        body = re.sub(r'<[^>]+>', ' ', body_m.group(1))[:800] if body_m else html[:500]

        return title, (desc + ' ' + body), final_url
    except Exception as e:
        return '', '', url

def build_entry(domain, url, original_url, query, rank, title, body):
    combined = title + ' ' + body
    is_sc = has_simplified_chinese(combined)
    sc_chars = '、'.join(get_simplified_hits(combined)) if is_sc else '—'
    flags = []
    if is_sc: flags.append('🈶 含簡中')
    if domain.endswith('.cn') or '.cn/' in url: flags.append('🇨🇳 .cn網域')
    if re.search(r'login|signin|verify|account|auth', url.lower()): flags.append('🔑 仿冒登入')
    if re.search(r'download|install|apk|setup', url.lower()): flags.append('📦 假下載')
    return {
        'domain': domain,
        'url': url,
        'original_url': original_url,
        'query': query,
        'rank': rank,
        'title': title[:80],
        'simplified_chars': sc_chars,
        'flags': ' '.join(flags) if flags else '⚠ 偽冒網域',
        'date': datetime.now().strftime('%Y-%m-%d'),
    }

print('✅ 設定完成')
print(f'📋 搜尋關鍵字：{len(SEARCH_QUERIES)} 組，每組前 {RESULTS_PER_QUERY} 筆')
print('🎯 篩選條件：網域含「line」且非官方白名單 → 直接列入')
print('🈶 簡體中文：額外標記（非必要條件）')

# ── 測試：驗證 linebbn.com 能被偵測到
test = 'linebbn.com'
print(f'\n🧪 測試 {test}：', end=' ')
print('✅ 會被列入' if (is_line_spoof(test) and not is_official(test)) else '❌ 不會被列入')


In [ ]:
# Cell 3｜🔍 用真實瀏覽器模擬 Google 搜尋
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import urllib.parse, time, re, random

# 初始化所有變數
all_urls = {}
non_official = {}
results = []
skipped = []
failed = []

def create_driver():
    """建立 Chrome 瀏覽器，自動偵測 Colab 環境"""
    opts = Options()
    opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument('--window-size=1280,800')
    opts.add_argument('--lang=zh-TW')
    opts.add_argument('--remote-debugging-port=9222')
    opts.add_argument(
        '--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_experimental_option('useAutomationExtension', False)

    # 依序嘗試幾種啟動方式
    driver = None
    errors = []

    # 方式 1：webdriver-manager 自動下載對應版本
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=opts)
        print('  ✅ ChromeDriver (webdriver-manager)')
        return driver
    except Exception as e:
        errors.append(f'webdriver-manager: {e}')

    # 方式 2：Colab 內建路徑
    for path in ['/usr/bin/chromedriver', '/usr/local/bin/chromedriver']:
        try:
            service = Service(path)
            driver = webdriver.Chrome(service=service, options=opts)
            print(f'  ✅ ChromeDriver ({path})')
            return driver
        except Exception as e:
            errors.append(f'{path}: {e}')

    # 方式 3：不指定 service，讓 selenium 自動尋找
    try:
        driver = webdriver.Chrome(options=opts)
        print('  ✅ ChromeDriver (自動尋找)')
        return driver
    except Exception as e:
        errors.append(f'auto: {e}')

    raise RuntimeError('無法啟動 ChromeDriver，請執行 Cell 1 重新安裝\n' + '\n'.join(errors))

def parse_google_results(html):
    """從 Google HTML 解析搜尋結果 URL"""
    soup = BeautifulSoup(html, 'lxml')
    found = []
    seen_domains = set()

    # 方法 1：/url?q= 格式（主要方式）
    for a in soup.select('a[href^="/url?q="]'):
        qs = urllib.parse.parse_qs(urllib.parse.urlparse(a['href']).query)
        real = qs.get('q', [''])[0]
        if real.startswith('http') and 'google.com' not in real:
            d = extract_domain(real)
            if d and d not in seen_domains:
                found.append((real, d))
                seen_domains.add(d)

    # 方法 2：jsname 屬性的連結（新版 Google）
    if not found:
        for a in soup.find_all('a', href=True):
            href = a['href']
            if href.startswith('http') and 'google' not in href and 'gstatic' not in href:
                d = extract_domain(href)
                if d and d not in seen_domains and '.' in d:
                    found.append((href, d))
                    seen_domains.add(d)

    # 方法 3：cite 標籤備援
    if not found:
        for cite in soup.select('cite'):
            text = cite.get_text().split('›')[0].strip()
            text = re.sub(r'^https?://', '', text).rstrip('/')
            if '.' in text and 'google' not in text:
                url = 'https://' + text
                d = extract_domain(url)
                if d and d not in seen_domains:
                    found.append((url, d))
                    seen_domains.add(d)
    return found

def google_search_selenium(driver, query, num=20):
    found = []
    start = 0
    while len(found) < num:
        params = urllib.parse.urlencode({
            'q': query, 'hl': 'zh-TW', 'gl': 'tw',
            'num': 10, 'start': start,
        })
        try:
            driver.get(f'https://www.google.com/search?{params}')
            time.sleep(random.uniform(2.5, 4.5))

            # 偵測 CAPTCHA
            if 'sorry' in driver.current_url or 'captcha' in driver.page_source.lower()[:2000]:
                print('    ⚠ 偵測到 CAPTCHA，等待 20 秒...')
                time.sleep(20)
                driver.get(f'https://www.google.com/search?{params}')
                time.sleep(3)

            new_items = parse_google_results(driver.page_source)
            existing = {x[1] for x in found}
            new_items = [(u, d) for u, d in new_items if d not in existing]
            if not new_items:
                break
            found.extend(new_items)
            start += 10
            if start >= num:
                break
            time.sleep(random.uniform(2, 3))
        except Exception as e:
            print(f'    ✗ 錯誤：{e}')
            break
    return found[:num]

# ── 啟動瀏覽器
print('🌐 啟動 Chrome 瀏覽器...')
driver = create_driver()
# 隱藏 webdriver 特徵
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})
print('✅ 瀏覽器就緒')

print('\n' + '=' * 58)
print('  模擬 Google 搜尋 LINE 下載')
print('=' * 58)

try:
    for query in SEARCH_QUERIES:
        print(f'\n🔎 搜尋：「{query}」')
        items = google_search_selenium(driver, query, num=RESULTS_PER_QUERY)
        if not items:
            print('  ⚠ 未取得結果')
            continue
        for rank, (url, domain) in enumerate(items, 1):
            if is_official(domain):       label = '✅ 官方 '
            elif is_line_spoof(domain):   label = '🚨 偽冒'
            else:                         label = '⚠  其他'
            print(f'  #{rank:2d} {label}  {domain[:52]}')
            if url not in all_urls:
                all_urls[url] = {'query': query, 'rank': rank, 'domain': domain}
        time.sleep(random.uniform(3, 5))
finally:
    driver.quit()
    print('\n🌐 瀏覽器已關閉')

non_official = {u: m for u, m in all_urls.items() if not is_official(m['domain'])}
spoofs_found = {u: m for u, m in non_official.items() if is_line_spoof(m['domain'])}

print(f'\n{"="*58}')
print(f'  總計 URL        ：{len(all_urls)}')
print(f'  ✅ 官方網域      ：{len(all_urls) - len(non_official)}')
print(f'  ⚠  其他非官方   ：{len(non_official) - len(spoofs_found)}')
print(f'  🚨 偽冒 LINE 網域：{len(spoofs_found)}')
if spoofs_found:
    print(f'\n  偵測到的偽冒站：')
    for u, m in spoofs_found.items():
        print(f'    → {m["domain"]}  (排名 #{m["rank"]}，關鍵字：{m["query"]})')
print(f'{"="*58}')
print('\n✅ 搜尋完成，請執行 Cell 4')


In [ ]:
# Cell 4｜🔎 訪問非官方網站，確認並分類

if 'non_official' not in dir() or not isinstance(non_official, dict):
    print('⚠ 請先執行 Cell 3')
elif len(non_official) == 0:
    print('⚠ 無非官方網址，請重新執行 Cell 3')
else:
    results = []
    skipped = []
    failed = []

    print(f'掃描 {len(non_official)} 個非官方網址...\n')

    for i, (url, meta) in enumerate(non_official.items(), 1):
        domain = meta['domain']
        is_spoof = is_line_spoof(domain)
        mark = '🚨' if is_spoof else '—'
        print(f'[{i:2d}/{len(non_official)}] {mark} {domain[:55]}...', end=' ')

        # 含 line 字眼 → 不管有沒有簡中都列入
        if is_spoof:
            title, body, final_url = fetch_page(url)
            if not title and not body:
                print('（無法訪問，仍列入）')
                # 無法訪問但網域可疑，仍列入黑名單
                results.append(build_entry(
                    domain, url, url,
                    meta['query'], meta['rank'],
                    '', ''
                ))
            else:
                combined = title + ' ' + body
                is_sc = has_simplified_chinese(combined)
                sc_note = f'🈶 {"、".join(get_simplified_hits(combined))}' if is_sc else '（非簡中）'
                print(f'列入 {sc_note}')
                results.append(build_entry(
                    domain, final_url, url,
                    meta['query'], meta['rank'],
                    title, body
                ))
        else:
            # 不含 line → 跳過
            print('跳過（無 line 字眼）')
            skipped.append(domain)

        time.sleep(0.5)

    # 去重（同網域保留排名最前）
    seen = {}
    for r in sorted(results, key=lambda x: x['rank']):
        if r['domain'] not in seen:
            seen[r['domain']] = r
    results = list(seen.values())

    sc_count = sum(1 for r in results if r['simplified_chars'] != '—')

    print(f'\n{"="*55}')
    print(f'  🚨 偽冒 LINE 網站（含 line 字眼）：{len(results)} 個')
    print(f'     其中含簡體中文                ：{sc_count} 個')
    print(f'  — 跳過（無 line 字眼）           ：{len(skipped)} 個')
    print(f'{"="*55}')
    print('\n✅ 掃描完成，請執行 Cell 5 輸出')


In [ ]:
# Cell 5｜💾 產生輸出檔案

if 'results' not in dir() or not isinstance(results, list):
    print('⚠ 請先依序執行 Cell 3 → Cell 4')
elif len(results) == 0:
    print('ℹ 本次搜尋未找到偽冒站')
else:
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    total = len(results)

    # 1. 網域黑名單
    domain_file = OUTPUT_DIR / 'blacklist_domains.txt'
    with open(domain_file, 'w', encoding='utf-8') as f:
        f.write('# LINE 偽冒下載站黑名單\n')
        f.write(f'# 產生時間：{now}  總計：{total} 筆\n\n')
        for r in results:
            f.write(r['domain'] + '\n')

    # 2. URL 黑名單
    url_file = OUTPUT_DIR / 'blacklist_urls.txt'
    with open(url_file, 'w', encoding='utf-8') as f:
        f.write(f'# LINE 偽冒下載站黑名單（URL）\n# 產生時間：{now}\n\n')
        for r in results:
            f.write(r['url'] + '\n')

    # 3. CSV
    csv_file = OUTPUT_DIR / 'blacklist_report.csv'
    with open(csv_file, 'w', newline='', encoding='utf-8-sig') as f:
        fields = ['domain','url','query','rank','title','simplified_chars','flags','date','original_url']
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(results)

    # 4. HTML 報告（用字串拼接，避免多行 f-string 問題）
    html_file = OUTPUT_DIR / 'blacklist_report.html'

    # 產生資料列
    rows_html = ''
    for r in results:
        rows_html += (
            '<tr>'
            f'<td style="color:#ff6b6b"><code>{r["domain"]}</code></td>'
            f'<td style="color:#e3b341;font-size:12px">{r["title"]}</td>'
            f'<td style="font-size:11px;word-break:break-all;color:#8b949e">{r["url"]}</td>'
            f'<td style="color:#58a6ff;font-size:12px">{r["query"]}</td>'
            f'<td style="text-align:center">#{r["rank"]}</td>'
            f'<td style="color:#ff4444">{r["simplified_chars"]}</td>'
            f'<td style="color:#58a6ff;font-size:11px">{r["flags"]}</td>'
            f'<td style="color:#8b949e;font-size:12px">{r["date"]}</td>'
            '</tr>'
        )

    # CSS（不用雙大括號，改用變數）
    css = (
        'body{font-family:monospace;background:#0d1117;color:#c9d1d9;padding:24px;margin:0}'
        'h1{color:#ff4444;margin-bottom:4px}'
        'h2{color:#8b949e;font-size:13px;font-weight:normal;margin-top:4px}'
        '.badge{display:inline-block;padding:3px 10px;border-radius:12px;font-size:11px;margin:2px}'
        '.b-red{background:#3d1a1a;border:1px solid #ff4444;color:#ff6b6b}'
        '.b-yellow{background:#2d2500;border:1px solid #e3b341;color:#e3b341}'
        'table{width:100%;border-collapse:collapse;font-size:12px;margin-top:20px}'
        'th{background:#161b22;color:#8b949e;padding:10px 8px;text-align:left;border-bottom:2px solid #30363d;white-space:nowrap}'
        'td{padding:8px;border-bottom:1px solid #21262d;vertical-align:top}'
        'tr:hover td{background:#161b22}'
        'code{background:#1f2428;padding:2px 6px;border-radius:3px}'
    )

    html = (
        '<!DOCTYPE html><html lang="zh-TW"><head><meta charset="UTF-8">'
        f'<title>LINE 偽冒站黑名單 {now[:10]}</title>'
        f'<style>{css}</style></head><body>'
        '<h1>&#x1F6A8; LINE 偽冒下載站黑名單</h1>'
        f'<h2>來源：模擬 Google 搜尋「LINE 下載」｜非官方網域｜產生時間：{now}</h2>'
        '<div style="margin:12px 0">'
        f'<span class="badge b-red">&#x26A0; 偽冒站 {total} 個</span>'
        '<span class="badge b-yellow">&#x1F236; 頁面含簡體中文</span>'
        '<span class="badge b-red">&#x1F50D; 來自 Google 搜尋結果</span>'
        '</div>'
        '<table>'
        '<tr><th>偽冒網域</th><th>頁面標題</th><th>URL</th>'
        '<th>搜尋關鍵字</th><th>排名</th><th>簡中字</th><th>特徵</th><th>日期</th></tr>'
        f'{rows_html}'
        '</table></body></html>'
    )
    html_file.write_text(html, encoding='utf-8')

    print(f'\n✅ 輸出完成！偽冒站共 {total} 筆')
    for f in sorted(OUTPUT_DIR.iterdir()):
        print(f'   📄 {f.name:40s} {f.stat().st_size:,} bytes')


In [ ]:
# Cell 6｜📥 下載所有檔案（ZIP）
import shutil
from google.colab import files

if not any(OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else True:
    print('⚠ 尚無輸出檔案，請先執行 Cell 5')
else:
    zip_name = f'LINE_phishing_{datetime.now().strftime("%Y%m%d_%H%M")}'
    zip_path = f'/content/{zip_name}.zip'
    shutil.make_archive(f'/content/{zip_name}', 'zip', OUTPUT_DIR)
    print(f'📦 已打包：{zip_path}')
    files.download(zip_path)


In [ ]:
# Cell 7｜📊 Colab 內預覽報告

if 'results' not in dir() or not isinstance(results, list) or len(results) == 0:
    print('⚠ 無資料，請先依序執行 Cell 3 → Cell 4')
else:
    _all = len(all_urls) if 'all_urls' in dir() else '?'
    _official = (len(all_urls) - len(non_official)) if 'all_urls' in dir() and 'non_official' in dir() else '?'
    _non = len(non_official) if 'non_official' in dir() else '?'

    display(HTML(f'''
    <div style="display:flex;gap:12px;margin:16px 0;font-family:monospace">
      <div style="background:#1a0d0d;border:1px solid #ff4444;border-radius:8px;padding:14px 24px;text-align:center">
        <div style="font-size:36px;font-weight:bold;color:#ff4444">{len(results)}</div>
        <div style="font-size:11px;color:#8b949e">🚨 偽冒站</div></div>
      <div style="background:#1a0d0d;border:1px solid #e3b341;border-radius:8px;padding:14px 24px;text-align:center">
        <div style="font-size:36px;font-weight:bold;color:#e3b341">{_non}</div>
        <div style="font-size:11px;color:#8b949e">⚠ 非官方網址</div></div>
      <div style="background:#0d1a0d;border:1px solid #3fb950;border-radius:8px;padding:14px 24px;text-align:center">
        <div style="font-size:36px;font-weight:bold;color:#3fb950">{_official}</div>
        <div style="font-size:11px;color:#8b949e">✅ 官方網域</div></div>
    </div>'''))

    rows = ''.join(
        f'<tr style="border-bottom:1px solid #21262d">'
        f'<td style="padding:7px 8px"><code style="background:#1f2428;padding:2px 6px;border-radius:3px;color:#ff6b6b">{r["domain"]}</code></td>'
        f'<td style="padding:7px 8px;color:#e3b341;font-size:12px">{r["title"][:40]}</td>'
        f'<td style="padding:7px 8px;color:#58a6ff;font-size:12px">{r["query"]}</td>'
        f'<td style="padding:7px 8px;text-align:center">#{r["rank"]}</td>'
        f'<td style="padding:7px 8px;color:#ff4444;font-size:13px">{r["simplified_chars"]}</td>'
        f'</tr>'
        for r in results
    )
    display(HTML(f'''
    <div style="font-family:monospace;font-size:12px">
    <h3 style="color:#ff4444">🚨 偽冒站清單（含簡中）</h3>
    <table style="width:100%;border-collapse:collapse;background:#0d1117;color:#c9d1d9">
    <tr style="background:#161b22">
      <th style="padding:8px">偽冒網域</th>
      <th style="padding:8px">頁面標題</th>
      <th style="padding:8px">搜尋關鍵字</th>
      <th style="padding:8px">排名</th>
      <th style="padding:8px">簡中命中字</th></tr>
    {rows}</table></div>'''))


---
## 📝 使用注意事項

- **Google 限速**：搜尋間已加入延遲，若出現 `429` 錯誤請稍等幾分鐘後重試
- **搜尋關鍵字**：可在 Cell 2 的 `SEARCH_QUERIES` 清單自行新增
- **白名單調整**：在 `OFFICIAL_DOMAINS` 新增你信任的網域
- **定期執行**：每週執行一次，更新黑名單後匯入防火牆

**匯入防火牆/DNS：**
```bash
# Pi-hole
cp blacklist_domains.txt /etc/pihole/custom.list && pihole restartdns

# Squid Proxy
acl line_phish dstdomain "/etc/squid/blacklist_domains.txt"
http_access deny line_phish
```

⚠ **免責聲明**：本工具僅供企業內部資安防護使用。如遭詐騙請撥打 **165 反詐騙專線**。
